# Phase 11C — Three training runs × five grouped folds for legacy DNABERT and DNABERT-2

**Purpose:** measure seed-to-seed variability separately from fold-to-fold variability on the locked Stage 1 and Stage 2 similarity-grouped splits. Phase 2.1 supplies the existing training run; this notebook trains **two additional independent seeds** for both transformer models on both stages. It leaves Phases 1–11B untouched.

### Kaggle inputs, setup, and four resumable runs

Attach these two saved inputs for every run (ZIP file or Kaggle-unpacked output):

1. `[1] enhancer_audit_v1_outputs.zip`
2. `[2.1] enhancer_phase2_corrected_results.zip`

Use a **GPU** and turn **Internet on** for pinned pretrained model downloads. The notebook pins `transformers==4.44.2`. At the top of Cell 0 set `SHARD_ID` to 0, 1, 2, then 3 in separate Kaggle runs:

| Shard | Model | Stage | Additional fits |
|---|---|---|---:|
| 0 | DNABERT-2 | 1 | 2 seeds × 5 folds |
| 1 | DNABERT-2 | 2 | 2 seeds × 5 folds |
| 2 | legacy DNABERT | 1 | 2 seeds × 5 folds |
| 3 | legacy DNABERT | 2 | 2 seeds × 5 folds |

For shards 1–3, also attach the **most recent `enhancer_phase11c_results_shard*.zip`** as a Kaggle Input. Each successive results ZIP carries all completed prediction files and hashes; only one previous results ZIP is needed. Each run writes a **small** `enhancer_phase11c_results_shardN.zip`, which is the **only file you need to download and attach for the next shard**. By default, `.pt` checkpoints stay in `/kaggle/working/enhancer_phase11c_work/weights/` for the current Kaggle session; they are **not** copied into the result ZIP or a multi-GB weights ZIP. Later sessions do not need those checkpoints for seed analysis. If you have room and explicitly want permanent weights, set `PACKAGE_WEIGHTS=True` in Cell 0 to produce a separate large archive. To resume an interrupted run **of the same shard**, the saved notebook Output must also contain its existing `.pt` weights. If a session ended before any results manifest was saved, rerun that shard from the beginning.

These are resource-intensive fits. The training protocol exactly keeps the old 2 epochs, batch 8, accumulation 2, LR 2e-5, reverse-complement augmentation after splitting, encoder pooling, and unchanged five folds. The base Phase 2.1 result had `base_seed + 100*stage + fold`; both new runs use the same seed formula with different base seeds. The original run was already inspected; variability is descriptive, not a fresh prospective experiment. **No threshold is chosen here.** Valid inner-validation threshold selection needs additional fitting inside each outer training fold and will be handled separately if justified after inspecting these seed results.

In [ ]:
# 0 — User-editable shard switch; protocol lock; prerequisites.
SHARD_ID=0   # Change to 0,1,2,3 in four Kaggle runs. Do not change the protocol below.
PACKAGE_WEIGHTS=False  # Optional archival ZIP; keep False to avoid a multi-GB download.
import os,sys,io,json,zipfile,hashlib,random,subprocess,gc,shutil,platform,time
from pathlib import Path
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY','1')
import numpy as np,pandas as pd
ROOT=Path('/kaggle/input');assert ROOT.is_dir(),'Attach Phase 1 and Phase 2.1 saved Outputs.'
WORK=Path('/kaggle/working/enhancer_phase11c_work');WORK.mkdir(parents=True,exist_ok=True)
PRED=WORK/'predictions';PRED.mkdir(exist_ok=True)
WEIGHTS=WORK/'weights';WEIGHTS.mkdir(exist_ok=True)
SHARDS=[('dnabert2',1),('dnabert2',2),('dnabert_legacy',1),('dnabert_legacy',2)]
assert SHARD_ID in range(4)
CURRENT_MODEL,CURRENT_STAGE=SHARDS[SHARD_ID]
EXPECTED={
 'stage1_candidate_manifest.csv':'e295155300fafc34528bc1868c6fad6ec78f3912c5b07e034710e81c3b4ac4b3',
 'stage2_candidate_manifest.csv':'e25291665c8c42522b662855d3774a0e425e37f603c2ab1f22f4a2c241def952',
 'audit_run_summary.json':'39ef44654b5576f4e71491f96e14e6816d0970a0a49ae3581d51cfb0f05a3263',
 'correction_report.json':'8296e5703c8e73765b42923be970b6752061639331c1318c6a3414767ed6914a',
 'all_oof_predictions.csv':'08d5d685690ec27bb36fdf6e7bbf8e3ff8dc7619484f9b3b51be54ff213e871f',
 'model_revisions.json':'f61445f06d5360c6791faef02a074dcf55bab191a9d4e1bce2c6e11c4a90cdc0'}
MODEL_IDS={'dnabert_legacy':'zhihan1996/DNA_bert_6',
 'dnabert2':'zhihan1996/DNABERT-2-117M'}
REVISIONS={'dnabert_legacy':'c56e67ea5827e0ddc67ef059addcf71569b1216e',
 'dnabert2':'ec1f874253852eb3907081f57294991b4280ceb6'}
CONFIG={'protocol':'phase11c_multiseed_grouped5_v1',
 'source_phase2_config_hash':'63e47ce62c02127cc6bf71762c1205d3bf473cc84085cae3f455ceffb7b1e5a3',
 'input_member_sha256':EXPECTED,'model_ids':MODEL_IDS,'revisions':REVISIONS,
 'base_seeds':[20260928,20261016,20261017],
 'new_base_seeds':[20261016,20261017],
 'models':['dnabert2','dnabert_legacy'],'stages':[1,2],'folds':[0,1,2,3,4],
 'epochs':2,'batch_size':8,'gradient_accumulation':2,'lr':2e-5,
 'weight_decay':.01,'gradient_clip':1.0,'dropout':.1,
 'rc_augment_train':True,'max_tokens':256,'transformers':'4.44.2',
 'score_orientation':'original sequence only','save_full_precision_weights':True}
def sha(raw):return hashlib.sha256(raw).hexdigest()
def file_sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 return h.hexdigest()
LOCK=sha(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode())
print('Protocol:',LOCK,'| SHARD_ID',SHARD_ID,'| model/stage',CURRENT_MODEL,CURRENT_STAGE)

In [ ]:
# 1 — Read exact member bytes from Kaggle-unpacked datasets or their ZIPs.
ZIPFILES=sorted(p for p in ROOT.rglob('*.zip') if p.is_file())
def all_members(name):
 for p in ROOT.rglob(name):
  if p.is_file():yield p.read_bytes(),str(p)
 for p in ZIPFILES:
  try:
   with zipfile.ZipFile(p) as z:
    for m in z.namelist():
     if m==name or m.endswith('/'+name):yield z.read(m),str(p)+'!'+m
  except (zipfile.BadZipFile,OSError):continue
def read_locked(name):
 good=[];bad=[]
 for raw,source in all_members(name):
  if sha(raw)==EXPECTED[name]:good.append((raw,source))
  else:bad.append(source)
 if not good:raise FileNotFoundError(f'Missing or wrong-version {name}. Unmatched candidates: {bad[:6]}')
 assert len({sha(raw) for raw,_ in good})==1
 print('VERIFIED',name,'from',good[0][1]);return good[0][0]
RAW={n:read_locked(n) for n in EXPECTED}
summary=json.loads(RAW['audit_run_summary.json'])
corr=json.loads(RAW['correction_report.json'])
revisions=json.loads(RAW['model_revisions.json'])
assert revisions['dnabert2']==REVISIONS['dnabert2']
assert revisions['dnabert_legacy']==REVISIONS['dnabert_legacy']
assert corr['config_hash']==CONFIG['source_phase2_config_hash']
assert corr['output_sha256']['all_oof_predictions.csv']==EXPECTED['all_oof_predictions.csv']
for s in (1,2):
 n=f'stage{s}_candidate_manifest.csv'
 assert summary['outputs_sha256'][n]==EXPECTED[n]
 assert corr['input_manifest_sha256'][n]==EXPECTED[n]
data={s:pd.read_csv(io.BytesIO(RAW[f'stage{s}_candidate_manifest.csv']),
 dtype=str,keep_default_na=False) for s in (1,2)}
for s,d in data.items():
 assert len(d)=={1:3268,2:1584}[s] and d.record_id.is_unique
 assert d.record_id.eq(d.sequence_sha256.str[:16]).all()
 assert d.sequence.str.fullmatch('[ACGT]{200}').all()
 assert all(sha(seq.encode())==digest for seq,digest in zip(d.sequence,d.sequence_sha256))
 d=d[d.role=='train_candidate'].copy().reset_index(drop=True)
 assert len(d)=={1:2838,2:1376}[s]
 assert set(d.label)=={'0','1'} and set(d.fold_5)==set('01234')
 assert d.groupby('similarity_group_id').fold_5.nunique().max()==1
 data[s]=d
BASE=pd.read_csv(io.BytesIO(RAW['all_oof_predictions.csv']),dtype={'record_id':str,'similarity_group_id':str})
assert len(BASE)==37926 and BASE.config_hash.eq(CONFIG['source_phase2_config_hash']).all()
assert not BASE.duplicated(['stage','model','record_id']).any()
for s in (1,2):
 truth=data[s].set_index('record_id')
 for model in ('finetuned_dnabert2','finetuned_dnabert_legacy'):
  p=BASE[(BASE.stage==s)&(BASE.model==model)]
  assert len(p)==len(truth) and set(p.record_id)==set(truth.index)
  t=truth.loc[p.record_id]
  assert p.fold_5.astype(str).tolist()==t.fold_5.tolist()
  assert p.label.astype(str).tolist()==t.label.tolist()
  assert p.similarity_group_id.tolist()==t.similarity_group_id.tolist()
print('Locked Stage 1/2 manifests and original-seed OOF predictions verified.')

In [ ]:
# 2 — Recover prior results either as a ZIP or a Kaggle-unpacked directory.
import re
PRIOR=[]
for p in ZIPFILES:
 if re.search(r'enhancer_phase11c_results_shard[0-3]\.zip$',p.name):
  with zipfile.ZipFile(p) as z:
   assert z.testzip() is None
   report=json.loads(z.read('phase11c_run_manifest.json'))
  PRIOR.append(('zip',p,report))
for p in ROOT.rglob('phase11c_run_manifest.json'):
 if p.is_file():PRIOR.append(('folder',p.parent,json.loads(p.read_text())))
PRIOR=[item for item in PRIOR if item[2].get('protocol_sha256')==LOCK]
if PRIOR:
 PRIOR.sort(key=lambda x:len(x[2]['completed_jobs']))
 kind,source,report=PRIOR[-1]
 # Ties must contain the same result index, regardless of ZIP/unpacked form.
 for _,_,other in PRIOR:
  if len(other['completed_jobs'])==len(report['completed_jobs']):
   assert other['completed_jobs']==report['completed_jobs'],'Conflicting prior result versions.'
 job_index={}
 for key,meta in report['completed_jobs'].items():
  if kind=='zip':
   with zipfile.ZipFile(source) as z:raw=z.read(meta['prediction_filename'])
  else:raw=(source/meta['prediction_filename']).read_bytes()
  assert sha(raw)==meta['prediction_sha256'],key
  dest=PRED/Path(meta['prediction_filename']).name
  if dest.exists():assert sha(dest.read_bytes())==sha(raw)
  else:dest.write_bytes(raw)
  job_index[key]=meta
 print('Recovered',len(job_index),'new-seed fits from',source)
else:
 job_index={}
 print('No previous Phase 11C results attached.')
assert SHARD_ID*10<=len(job_index)<=(SHARD_ID+1)*10, (
 'For shard N attach the latest completed results from shard N-1, or same-shard partial results.')
assert all(k.split('__')[0] in CONFIG['models'] for k in job_index)
for key,meta in job_index.items():
 if meta.get('checkpoint_shard')!=SHARD_ID:continue
 dest=WEIGHTS/Path(meta['checkpoint_filename']).name
 if dest.is_file() and file_sha(dest)==meta['checkpoint_sha256']:continue
 found=False
 for p in ROOT.rglob(dest.name):
  if p.is_file() and file_sha(p)==meta['checkpoint_sha256']:
   shutil.copy2(p,dest);found=True;break
 if not found:
  for archive in ZIPFILES:
   if not re.search(r'enhancer_phase11c_weights_shard[0-3]\.zip$',archive.name):continue
   with zipfile.ZipFile(archive) as z:
    if meta['checkpoint_filename'] not in z.namelist():continue
    with z.open(meta['checkpoint_filename']) as src,open(dest,'wb') as out:
     shutil.copyfileobj(src,out)
   if file_sha(dest)==meta['checkpoint_sha256']:found=True;break
 if not found:raise FileNotFoundError('To resume the same shard, attach its saved weights for '+key)
print('This run will train only missing jobs for shard',SHARD_ID)

In [ ]:
# 3 — Pin model dependencies and implement the exact historical encoder/head/augmentation.
from importlib.metadata import version,PackageNotFoundError
try:installed=version('transformers')
except PackageNotFoundError:installed='missing'
if installed!=CONFIG['transformers']:
 assert 'transformers' not in sys.modules,'Restart Kaggle session; Transformers was already imported.'
 subprocess.run([sys.executable,'-m','pip','install','-q',
  'transformers=='+CONFIG['transformers']],check=True,timeout=900)
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import transformers
from transformers import AutoTokenizer,AutoModel
assert transformers.__version__==CONFIG['transformers']
assert torch.cuda.is_available(),'Enable a Kaggle GPU for 40 additional fine-tuning fits.'
DEVICE=torch.device('cuda')
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__)
RC=str.maketrans('ACGT','TGCA')
def revcomp(s):return s.translate(RC)[::-1]
def seed_everything(v):
 random.seed(v);np.random.seed(v);torch.manual_seed(v);torch.cuda.manual_seed_all(v)
 torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
def text_for(model,s):
 return ' '.join(s[i:i+6] for i in range(195)) if model=='dnabert_legacy' else s
def load_backbone(model):
 remote=MODEL_IDS[model];revision=REVISIONS[model]
 tok=AutoTokenizer.from_pretrained(remote,revision=revision,trust_remote_code=True)
 if model=='dnabert_legacy':
  from transformers.models.bert.configuration_bert import BertConfig
  cfg=BertConfig.from_pretrained(remote,revision=revision)
  enc=AutoModel.from_pretrained(remote,revision=revision,trust_remote_code=True,config=cfg)
 else:
  try:enc=AutoModel.from_pretrained(remote,revision=revision,trust_remote_code=True)
  except (ValueError,TypeError):
   from transformers.models.bert.configuration_bert import BertConfig
   cfg=BertConfig.from_pretrained(remote,revision=revision)
   enc=AutoModel.from_pretrained(remote,revision=revision,trust_remote_code=True,config=cfg)
  import importlib
  mod=importlib.import_module(type(enc).__module__)
  assert hasattr(mod,'BertUnpadSelfAttention') and hasattr(mod,'flash_attn_qkvpacked_func')
  mod.flash_attn_qkvpacked_func=None
 return tok,enc
def tokenize(tok,model,seqs):
 b=tok([text_for(model,s) for s in seqs],padding=True,
  add_special_tokens=True,truncation=False,return_tensors='pt')
 assert b['input_ids'].shape[1]<=CONFIG['max_tokens'],(model,b['input_ids'].shape)
 return {k:v.to(DEVICE) for k,v in b.items() if k in ('input_ids','attention_mask')}
class BinaryEncoder(nn.Module):
 def __init__(self,enc):
  super().__init__();self.encoder=enc;self.dropout=nn.Dropout(CONFIG['dropout'])
  self.head=nn.Linear(int(enc.config.hidden_size),2)
 def forward(self,b):
  output=self.encoder(**b)
  h=getattr(output,'last_hidden_state',None)
  if h is None:h=output[0]
  mask=b['attention_mask'].to(h.dtype).unsqueeze(-1)
  pooled=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
  return self.head(self.dropout(pooled))
class Rows(Dataset):
 def __init__(self,seqs,y):self.seqs=list(seqs);self.y=list(y)
 def __len__(self):return len(self.y)
 def __getitem__(self,i):return self.seqs[i],self.y[i]
def split(stage,fold):
 d=data[stage]
 tr=d[d.fold_5!=str(fold)].reset_index(drop=True)
 va=d[d.fold_5==str(fold)].reset_index(drop=True)
 assert not set(tr.similarity_group_id)&set(va.similarity_group_id)
 return tr,va

In [ ]:
# 4 — Train each pending seed/fold, save OOF scores and exact full-precision weights.
def train_job(model,stage,base_seed,fold):
 tr,va=split(stage,fold)
 effective_seed=base_seed+stage*100+fold
 seed_everything(effective_seed)
 tok,enc=load_backbone(model);net=BinaryEncoder(enc).to(DEVICE)
 seq=tr.sequence.tolist();labels=tr.label.astype(int).tolist()
 seq+=list(map(revcomp,tr.sequence.tolist()))
 labels+=tr.label.astype(int).tolist()
 ds=Rows(seq,labels)
 def collate(items):
  x=tokenize(tok,model,[s for s,_ in items])
  return x,torch.tensor([y for _,y in items],device=DEVICE,dtype=torch.long)
 opt=torch.optim.AdamW(net.parameters(),lr=CONFIG['lr'],weight_decay=CONFIG['weight_decay'])
 scaler=torch.amp.GradScaler('cuda',enabled=True)
 for epoch in range(CONFIG['epochs']):
  rng=torch.Generator().manual_seed(effective_seed+epoch)
  loader=DataLoader(ds,batch_size=CONFIG['batch_size'],shuffle=True,
   generator=rng,collate_fn=collate,num_workers=0)
  net.train();opt.zero_grad(set_to_none=True);losses=[]
  for i,(x,y) in enumerate(loader):
   with torch.autocast('cuda',dtype=torch.float16):
    loss=nn.functional.cross_entropy(net(x),y)
   assert torch.isfinite(loss),(model,stage,base_seed,fold,epoch,i)
   scaler.scale(loss/CONFIG['gradient_accumulation']).backward()
   if (i+1)%CONFIG['gradient_accumulation']==0 or i+1==len(loader):
    scaler.unscale_(opt);nn.utils.clip_grad_norm_(net.parameters(),CONFIG['gradient_clip'])
    scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
   losses.append(float(loss.detach().cpu()))
  print(model,'stage',stage,'seed',base_seed,'fold',fold,'epoch',epoch+1,
   'train loss',round(float(np.mean(losses)),5),flush=True)
 net.eval();scores=[];seqs=va.sequence.tolist()
 with torch.inference_mode():
  for i in range(0,len(seqs),CONFIG['batch_size']):
   x=tokenize(tok,model,seqs[i:i+CONFIG['batch_size']])
   with torch.autocast('cuda',dtype=torch.float16):
    p=torch.softmax(net(x).float(),dim=1)[:,1]
   scores.extend(p.cpu().numpy().tolist())
 scores=np.asarray(scores,dtype=float)
 assert len(scores)==len(va) and np.isfinite(scores).all() and ((scores>=0)&(scores<=1)).all()
 result=va[['record_id','similarity_group_id','label','fold_5']].copy()
 result['stage']=stage;result['model']='finetuned_'+model
 result['base_seed']=base_seed;result['score']=scores
 result['effective_seed']=effective_seed
 key=f'{model}__s{stage}__seed{base_seed}__fold{fold}'
 name=key+'.csv';outfile=PRED/name
 tmp=outfile.with_suffix('.tmp');result.to_csv(tmp,index=False);os.replace(tmp,outfile)
 # Save in full precision. The saved state, model revision, and matching source row hashes
 # are declared in the two artifact manifests; optimizer state is not required for inference.
 weight=WEIGHTS/(key+'.pt');wtmp=weight.with_suffix('.tmp')
 torch.save({'state_dict':{k:v.detach().cpu().contiguous() for k,v in net.state_dict().items()},
  'protocol_sha256':LOCK,'model_key':model,'model_revision':REVISIONS[model],
  'stage':stage,'base_seed':base_seed,'effective_seed':effective_seed,'fold':fold,
  'manifest_sha256':EXPECTED[f'stage{stage}_candidate_manifest.csv'],
  'epochs':CONFIG['epochs']},wtmp)
 os.replace(wtmp,weight)
 meta={'prediction_filename':'predictions/'+name,'prediction_sha256':file_sha(outfile),
  'checkpoint_filename':'weights/'+weight.name,'checkpoint_sha256':file_sha(weight),
  'model_key':model,'model_revision':REVISIONS[model],
  'stage':stage,'base_seed':base_seed,'effective_seed':effective_seed,'fold':fold,
  'rows':len(va),'checkpoint_shard':SHARD_ID}
 del net,enc,tok,ds,loader,opt,scaler;gc.collect();torch.cuda.empty_cache()
 return key,meta
for base_seed in CONFIG['new_base_seeds']:
 for fold in CONFIG['folds']:
  key=f'{CURRENT_MODEL}__s{CURRENT_STAGE}__seed{base_seed}__fold{fold}'
  if key in job_index:
   print('REUSED',key,flush=True);continue
  print('TRAINING',key,flush=True)
  job,meta=train_job(CURRENT_MODEL,CURRENT_STAGE,base_seed,fold)
  job_index[job]=meta
  (WORK/'progress.json').write_text(json.dumps({'protocol_sha256':LOCK,
   'last_completed':job,'completed_jobs':job_index},indent=2,sort_keys=True)+'\n')
  print('SAVED',job,'prediction + checkpoint',flush=True)
print('Shard training finished; total new-seed jobs available:',len(job_index),'of 40.')

In [ ]:
# 5 — Construct the complete three-seed table when all shards exist.
# Partial shards still package predictions. No threshold fitting uses OOF scores.
from sklearn.metrics import roc_auc_score,average_precision_score,brier_score_loss
all_new=[]
for key,meta in sorted(job_index.items()):
 p=PRED/Path(meta['prediction_filename']).name
 assert p.is_file() and file_sha(p)==meta['prediction_sha256']
 d=pd.read_csv(p,dtype={'record_id':str,'similarity_group_id':str,'model':str})
 expected=data[meta['stage']]
 expected=expected[expected.fold_5==str(meta['fold'])].reset_index(drop=True)
 assert d.record_id.tolist()==expected.record_id.tolist()
 assert d.label.astype(str).tolist()==expected.label.tolist()
 assert d.similarity_group_id.tolist()==expected.similarity_group_id.tolist()
 assert len(d)==meta['rows'] and d.score.between(0,1).all()
 all_new.append(d)
prior=BASE[BASE.model.isin(['finetuned_dnabert2','finetuned_dnabert_legacy'])].copy()
prior['base_seed']=CONFIG['base_seeds'][0]
prior['effective_seed']=CONFIG['base_seeds'][0]+100*prior.stage+prior.fold_5
columns=['record_id','similarity_group_id','label','fold_5','stage','model','base_seed','effective_seed','score']
combined=pd.concat([prior[columns]]+[d[columns] for d in all_new],ignore_index=True)
assert not combined.duplicated(['stage','model','base_seed','record_id']).any()
combined.to_csv(WORK/'phase11c_all_available_oof_predictions.csv',index=False)
if len(job_index)==40:
 rows=[]
 for (stage,model,seed,fold),g in combined.groupby(['stage','model','base_seed','fold_5']):
  yy=g.label.astype(int).to_numpy();ss=g.score.to_numpy(dtype=float)
  rows.append({'stage':stage,'model':model,'base_seed':seed,'fold_5':fold,'n':len(g),
   'auroc':roc_auc_score(yy,ss),
   'auprc':average_precision_score(yy,ss),
   'brier':brier_score_loss(yy,ss),
   'sensitivity_at_0_5':float((ss[yy==1]>=.5).mean()),
   'specificity_at_0_5':float((ss[yy==0]<.5).mean())})
 fm=pd.DataFrame(rows)
 assert len(fm)==2*2*3*5
 fm.to_csv(WORK/'phase11c_seed_fold_metrics.csv',index=False)
 summary=[]
 for (stage,model),g in fm.groupby(['stage','model']):
  foldmeans=g.groupby('fold_5').auroc.mean();seedmeans=g.groupby('base_seed').auroc.mean()
  summary.append({'stage':stage,'model':model,'mean_of_all_seed_fold_auroc':g.auroc.mean(),
   'fold_to_fold_sd_of_seed_means':foldmeans.std(ddof=1),
   'seed_to_seed_sd_of_fold_means':seedmeans.std(ddof=1),
   'minimum_seed_mean_auc':seedmeans.min(),'maximum_seed_mean_auc':seedmeans.max(),
   'mean_fold_brier':g.brier.mean(),
   'mean_fold_sensitivity_at_0_5':g.sensitivity_at_0_5.mean(),
   'n_seeds':3,'n_folds':5})
 pd.DataFrame(summary).to_csv(WORK/'phase11c_seed_vs_fold_variability.csv',index=False)
 wide=fm.pivot(index=['stage','base_seed','fold_5'],columns='model',values='auroc')
 baseline=[]
 for (stage,fold),g in BASE[BASE.model=='spaced6_4_logreg'].groupby(['stage','fold_5']):
  baseline.append((stage,fold,roc_auc_score(g.label,g.score)))
 baseline=dict(((s,f),value) for s,f,value in baseline)
 comparisons=[]
 for (stage,seed,fold),r in wide.iterrows():
  comparisons.append({'stage':stage,'base_seed':seed,'fold_5':fold,
   'dnabert2_minus_legacy':r['finetuned_dnabert2']-r['finetuned_dnabert_legacy'],
   'dnabert2_minus_fixed_spaced6_4':r['finetuned_dnabert2']-baseline[(stage,fold)]})
 pd.DataFrame(comparisons).to_csv(WORK/'phase11c_paired_fold_deltas.csv',index=False)
 print('COMPLETED all 40 new fits. Distinct fold and seed SDs:')
 print(pd.DataFrame(summary).to_string(index=False))
else:
 print('PARTIAL',len(job_index),'/40. Complete remaining SHARD_ID values with this results ZIP attached.')

In [ ]:
# 6 — Always save small cumulative results; package large weights only if requested.
result_files=sorted(PRED.glob('*.csv'))
manifest={'protocol_sha256':LOCK,'config':CONFIG,'current_shard':SHARD_ID,
 'completed_jobs':job_index,'new_jobs_completed':len(job_index),
 'expected_new_jobs':40,'complete':len(job_index)==40,
 'phase2_original_seed_reused':20260928,
 'weights_policy':('A separate weights ZIP is created only when PACKAGE_WEIGHTS=True; '
  'otherwise .pt files remain in the current Kaggle working directory. '
  'The cumulative results ZIP contains all prediction files but no .pt weights.'),
 'software':{'python':platform.python_version(),'torch':torch.__version__,
             'transformers':transformers.__version__,'numpy':np.__version__},
 'limitations':['Seed variation is descriptive; old seed was evaluated before selecting this analysis.',
  'No new similarity grouping or alternative LCS threshold is evaluated.',
  'No outer OOF score is used to choose a threshold.',
  'Fine-tuning seeds are independent initializations/shuffles on the same fixed folds; folds overlap in training data.']}
summary_files=['phase11c_all_available_oof_predictions.csv']
if manifest['complete']:summary_files+=['phase11c_seed_fold_metrics.csv',
 'phase11c_seed_vs_fold_variability.csv','phase11c_paired_fold_deltas.csv']
manifest['summary_sha256']={n:file_sha(WORK/n) for n in summary_files}
(WORK/'phase11c_run_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
result_zip=Path(f'/kaggle/working/enhancer_phase11c_results_shard{SHARD_ID}.zip')
with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in result_files:z.write(p,'predictions/'+p.name)
 for n in summary_files:z.write(WORK/n,n)
 z.write(WORK/'phase11c_run_manifest.json','phase11c_run_manifest.json')
with zipfile.ZipFile(result_zip) as z:
 assert z.testzip() is None
 for key,meta in job_index.items():
  assert sha(z.read(meta['prediction_filename']))==meta['prediction_sha256']
 for n,digest in manifest['summary_sha256'].items():assert sha(z.read(n))==digest
print('RESULTS ZIP:',result_zip,'SHA256',file_sha(result_zip),'size GB',round(result_zip.stat().st_size/1e9,3))
if PACKAGE_WEIGHTS:
 weights_for_shard=[(key,meta) for key,meta in job_index.items()
                    if meta.get('checkpoint_shard')==SHARD_ID]
 assert len(weights_for_shard)==10,'Current shard must have ten saved weights.'
 weights_manifest={'protocol_sha256':LOCK,'shard':SHARD_ID,
  'weights':{key:{'filename':meta['checkpoint_filename'],
                  'sha256':meta['checkpoint_sha256']} for key,meta in weights_for_shard}}
 wmanifest=WORK/'phase11c_weights_manifest.json'
 wmanifest.write_text(json.dumps(weights_manifest,indent=2,sort_keys=True)+'\n')
 weight_zip=Path(f'/kaggle/working/enhancer_phase11c_weights_shard{SHARD_ID}.zip')
 with zipfile.ZipFile(weight_zip,'w',zipfile.ZIP_STORED,allowZip64=True) as z:
  for key,meta in weights_for_shard:
   p=WEIGHTS/Path(meta['checkpoint_filename']).name
   assert p.is_file() and file_sha(p)==meta['checkpoint_sha256']
   z.write(p,meta['checkpoint_filename'])
  z.write(wmanifest,'phase11c_weights_manifest.json')
 with zipfile.ZipFile(weight_zip) as z:
  assert z.testzip() is None
  assert set(z.namelist())=={'phase11c_weights_manifest.json'}|{m['checkpoint_filename'] for _,m in weights_for_shard}
 print('OPTIONAL WEIGHTS ZIP:',weight_zip,'SHA256',file_sha(weight_zip))
else:
 print('No weights ZIP created. Full-precision .pt files remain only in',WEIGHTS)
print('Download the results ZIP. Attach it as an Input before the next SHARD_ID.')